## Fine-tuning

# Getting dependencies done

For working with LLMs, people may use `pytorch`, `tensorflow` and often rely on the `transformers` library to implement the actual architecture set up for large language models.

In [ ]:
%pip install transformers[torch] datasets

In [ ]:
import transformers
import torch

from transformers import AutoModel, AutoTokenizer
from datasets import load_dataset
from transformers import Trainer, TrainingArguments
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM

In [ ]:
transformers.enable_full_determinism( 0 ) ## this is supposed to help with reproducibility

## Define models and data for fine-tuning

Fine-tuning is conducted on an existing model, such as the _gpt2_.
This is not a top-notch model, but we try to keep things small and easy for this playful experiment.
We also define data used for training and validation.
In this case, we use the same data for both: you would not usually do this as this leads to overfitting – but as we intentially may want to make this model a bit over-the-top this is not critical. 

In [ ]:
model = "openai-community/gpt2"
block_size = 2**5 # this is small for demonstration purposes, you can increase it

In [ ]:
datasets = load_dataset("text", data_files={"train": './data/*.txt', "validation": './data/*.txt'})

In [ ]:
## Massage the datasets to be in the right format for training

We need to tokenize the data, that is, split it into smaller bits.
Following this, we organise them into blocks (groups) of spesific size to make them ready for learning.
The size of blocks, `block_size` was intentionally set as small to make the model a bit easier to work on.
You can increase it to make the model more accurate, but it will also take longer to train.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token

def tokenize_function(examples):
            return tokenizer(examples["text"])

tokenized_datasets = datasets.map(tokenize_function, batched=True, num_proc=4, remove_columns=["text"])


In [ ]:
def group_texts(examples):
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    total_length = (total_length // block_size) * block_size
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

lm_datasets = tokenized_datasets.map(
    group_texts,
    batched=True,
    batch_size=1000,
    num_proc=4
)

## Training the model

CausalLM is just a model that means we do text generation.
Training parameters set up, the most important are `num_train_epochs` (more is higher quality, but slower), similar with `learning_rate` which impacts how much the model learns at each evaluation round.
Then we train the model, defining what data goes in and use the pre-set arguments.

In [ ]:
model = AutoModelForCausalLM.from_pretrained(model)

In [ ]:
training_args = TrainingArguments(
    eval_strategy="epoch",
    learning_rate=2e-5,
    num_train_epochs=2, ## this is small for demonstration purposes, you can increase it
    weight_decay=0.01,
    output_dir= f"./models/finetuned",
    logging_dir= f"./models/finetuned-logs",
    logging_steps=10,
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_datasets["train"],
    eval_dataset=lm_datasets["validation"],
)

trainer.train()

In [ ]:
trainer.save_model(f"./models/finetuned")
tokenizer.save_pretrained(f"./models/finetuned")